In [17]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as opt
import utils
import hyp
from dqn import MolDQN, MolDQN_2
from rdkit import Chem
from rdkit.Chem import QED
from rdkit.Chem import AllChem
from rdkit import RDConfig
import os
import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer
from sklearn import preprocessing
from environment import Molecule
from rdkit import DataStructs
import pandas as pd
import pickle
import pdb
import time
import csv
from eval import EnsembleCalculator, load_models, to_numpy
from alfabet import model as bde_model

def ev2kcal_per_mol(ev):
    return ev * 23.0609

def calc_react_idx(data):
    ip = data['energy'][0] - data['energy'][1]
    ea = data['energy'][1] - data['energy'][2]
    f_el = data['charges'][1] - data['charges'][0]
    f_nuc = data['charges'][2] - data['charges'][1]
    chi = 0.5 * (ip + ea)
    eta = 0.5 * (ip - ea)
    omega = (chi ** 2) / (2 * eta)
    f_rad = 0.5 * (f_el + f_nuc)
    _omega = np.expand_dims(omega, axis=-1)
    omega_el = f_el * _omega
    omega_nuc = f_nuc * _omega
    omega_rad = f_rad * _omega
    return dict(ip=ip, ea=ea, f_el=f_el, f_nuc=f_nuc, f_rad=f_rad,
                chi=chi, eta=eta, omega=omega,
                omega_el=omega_el, omega_nuc=omega_nuc, omega_rad=omega_rad)


def get_scaler(path, real_col_id = 1):
    real = []
    # global real
    with open (path) as f:
        s = csv.reader(f, delimiter="\t");
        next(s)
        for r in s:
            if r[real_col_id] != '':
                real.append([float(r[real_col_id])])
    return preprocessing.MinMaxScaler().fit(real)

# device = torch.device("cuda:0")
device = torch.device("cpu")

class AimnetNseModel():
    """The original model is not pickable and can't be used with spawn. This class is a warpper of EnsembleCalculator."""
    def __init__(self, path, device):
        self.path = path
        self.device = device
        self.model = load_models([path]).to(device)

    def __setstate__(self, state):
        self.path = state['path']
        self.device = state['device']
        self.model = load_models([self.path]).to(device)

    def __getstate__(self):
        return dict(path = self.path, device = device)

ip_model_path = [
        'aimnetnse-models/aimnet-nse-cv0.jpt',
        'aimnetnse-models/aimnet-nse-cv1.jpt',
        'aimnetnse-models/aimnet-nse-cv2.jpt',
        'aimnetnse-models/aimnet-nse-cv3.jpt',
        'aimnetnse-models/aimnet-nse-cv4.jpt']
ip_model = [AimnetNseModel(ipmp, device) for ipmp in ip_model_path]

def rwmol2multi_data(mol, copyMol = True):
        
    mol = Chem.RWMol(mol)
    mol = Chem.AddHs(mol)
    # https://sourceforge.net/p/rdkit/mailman/message/33386856/
    # EmbedMolecule may return -1 for large mols.
    # cid = AllChem.EmbedMolecule(mol, useRandomCoords=mol.GetNumAtoms() > 96)
    # if cid == -1:
    try:
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=len(ip_model) * 2,useRandomCoords=True, maxAttempts=24)
        if len(cids) < len(ip_model):
            return dict(), False
        coords = [mol.GetConformer(cid).GetPositions() for cid in cids[0:len(ip_model)]]
        coords = [torch.tensor(coord, dtype=torch.float).unsqueeze(0).repeat(3, 1, 1).to(device) for coord in coords]
        numbers = [a.GetAtomicNum() for a in mol.GetAtoms()]
        numbers = torch.tensor(numbers, dtype=torch.long).unsqueeze(0).repeat(3, 1).to(device)
        charge = torch.tensor([1, 0, -1]).to(device)  # cation, neutral, anion
        mult = torch.tensor([2, 1, 2]).to(device)
        return [dict(coord=coord, numbers=numbers, charge=charge, mult=mult) for coord in coords], True
    except Exception as e:
        print("Exception: ")
        print(e)
        return dict(), False

In [31]:
s = "Oc1ccncc1O"
mol = Chem.MolFromSmiles(s)
mol = Chem.RWMol(mol)
mol = Chem.AddHs(mol)
cids = AllChem.EmbedMultipleConfs(mol, numConfs=len(ip_model) * 2,useRandomCoords=True, maxAttempts=24)
print(len(cids))

10


In [35]:
for i in range(5):
    coords = [mol.GetConformer(cid).GetPositions() for cid in cids[0:len(ip_model)]]
    coords = [torch.tensor(coord, dtype=torch.float).unsqueeze(0).repeat(3, 1, 1).to(device) for coord in coords]
    numbers = [a.GetAtomicNum() for a in mol.GetAtoms()]
    numbers = torch.tensor(numbers, dtype=torch.long).unsqueeze(0).repeat(3, 1).to(device)
    charge = torch.tensor([1, 0, -1]).to(device)  # cation, neutral, anion
    mult = torch.tensor([2, 1, 2]).to(device)
    datas = [dict(coord=coord, numbers=numbers, charge=charge, mult=mult) for coord in coords]
    pred_ip = 0.0
    last_data = None
    for im, data in zip(ip_model, datas):

        # disable optimizations for safety. with some combinations of pytorch/cuda it's getting very slow
        with torch.jit.optimized_execution(False), torch.no_grad():   
            pred = im.model(data)
        pred['charges'] = pred['charges'].sum(-1)
        pred = to_numpy(pred)
        # calculate indicies
        pred.update(calc_react_idx(pred))
        # write
        for k, v in pred.items():
            pred[k] = v.tolist()
        pred_ip = pred_ip + ev2kcal_per_mol( pred['ip'])
    print(pred_ip / len(ip_model))

206.0423705869377
206.0423705869377
206.0423705869377
206.0423705869377
206.0423705869377
